# Mango Leaf Disease Detection — Inception V3 Transfer Learning

This notebook trains the two models used by the production backend:

1. **Disease classifier** — Inception V3 (ImageNet-pretrained, transfer learning) fine-tuned on the
   [Mango Leaf Disease Dataset](https://www.kaggle.com/datasets/aryashah2k/mango-leaf-disease-dataset) (Kaggle).
2. **Leaf gate** — a small binary classifier (MobileNetV2 backbone) that determines whether an
   image is a mango leaf at all, before the disease classifier is trusted. A softmax classifier
   trained only on disease classes has no mechanism to express "none of the above"; without this
   gate, an unrelated photo (a different plant, an object, a blank frame) would still receive a
   confident-looking disease label.

Outputs of this notebook, copied into `backend/app/ml/`:

| File | Produced by |
|---|---|
| `model.tflite` | Disease classifier, converted from Keras `.h5` |
| `leaf_gate.tflite` | Binary gate classifier |
| `labels.json` | Class index to disease name mapping, plus both thresholds and both models' test accuracy |

**Why TFLite over ONNX for serving:** the backend is deployed on Render's free tier, which has a
tight memory ceiling. A full TensorFlow/Keras runtime in the serving container previously caused an
out-of-memory crash under load; the TFLite interpreter has a much smaller runtime footprint and
loads only the converted graph, not the training framework. ONNX export code is included later as
a documented alternative for a future ONNX Runtime-based host.

**Compatibility note:** TensorFlow 2.16 and later default `tf.keras` to Keras 3, under which
`TFLiteConverter.from_keras_model` is unreliable. The first code cell installs `tf-keras` and
forces the legacy Keras 2 engine before TensorFlow is imported, so the rest of the notebook —
model construction, training, and TFLite conversion — behaves identically to earlier TensorFlow
versions.

## 1. Environment setup

In [ ]:
!pip install -q tf-keras tensorflow-datasets "protobuf>=6.31.1" scikit-learn matplotlib seaborn kagglehub tf2onnx onnx pillow

In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import random
import pathlib

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input as inception_preprocess
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input as mobilenet_preprocess
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__, "(should be 2.x — TF_USE_LEGACY_KERAS redirects both `keras` and `tf.keras` to tf-keras)")
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Download the dataset

Uses `kagglehub`, which works in both Colab and Kaggle Notebooks without manual `kaggle.json`
handling — on Kaggle it authenticates automatically; on Colab it prompts for Kaggle credentials
the first time this cell runs.

The exact folder layout inside the downloaded archive is not assumed — `find_class_root` searches
for it directly and raises a clear error if the dataset's structure ever changes, instead of
failing with an unhelpful low-level exception.

In [ ]:
import kagglehub

DATASET_HANDLE = "aryashah2k/mango-leaf-disease-dataset"
dataset_root = kagglehub.dataset_download(DATASET_HANDLE)
print("Dataset downloaded to:", dataset_root)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def _has_image_files(directory):
    return any(f.suffix.lower() in IMAGE_EXTENSIONS for f in directory.iterdir() if f.is_file())


def find_class_root(root):
    root = pathlib.Path(root)
    best = None
    for candidate in [root, *root.rglob("*")]:
        if not candidate.is_dir():
            continue
        subdirs = [c for c in candidate.iterdir() if c.is_dir()]
        if len(subdirs) < 2:
            continue
        if all(_has_image_files(sd) for sd in subdirs):
            if best is None or len(subdirs) > len(best[1]):
                best = (candidate, subdirs)
    if best is None:
        raise FileNotFoundError(
            f"Could not find class subfolders under {root}. "
            "Inspect the downloaded dataset structure manually and adjust find_class_root."
        )
    return best[0]


data_dir = find_class_root(dataset_root)
class_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
CLASS_NAMES = [d.name for d in class_dirs]
print(f"Found {len(CLASS_NAMES)} classes under {data_dir}:", CLASS_NAMES)

## 3. Class balance

In [ ]:
counts = {d.name: len(list(d.glob("*"))) for d in class_dirs}
plt.figure(figsize=(10, 4))
sns.barplot(x=list(counts.keys()), y=list(counts.values()))
plt.xticks(rotation=45, ha="right")
plt.ylabel("Image count")
plt.title("Images per class")
plt.tight_layout()
plt.show()
print(counts)

## 4. Data pipelines

70/15/15 train/val/test split, Inception V3's expected 299x299 input, and augmentation applied to
the training set only — random flips, rotation, zoom, and contrast approximate real capture
variance (phone angle, lighting) without distorting the leaf texture that the disease signal
depends on.

In [ ]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", class_names=CLASS_NAMES,
)
raw_val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.3, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", class_names=CLASS_NAMES,
)

val_batches = raw_val_test_ds.cardinality()
raw_val_ds = raw_val_test_ds.take(val_batches // 2)
raw_test_ds = raw_val_test_ds.skip(val_batches // 2)

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])


def prep_train(x, y):
    return inception_preprocess(augment(x)), y


def prep_eval(x, y):
    return inception_preprocess(x), y


AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = raw_val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = raw_test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

NUM_CLASSES = len(CLASS_NAMES)
print("Classes:", NUM_CLASSES)

## 5. Disease classifier — Inception V3 transfer learning

Training proceeds in two phases:
1. **Head-only** — the Inception V3 backbone is frozen entirely; only the new classification head
   is trained, at a relatively high learning rate, until it converges.
2. **Fine-tune** — the top block of the backbone is unfrozen and training continues at a much
   lower learning rate, allowing the pretrained filters to adapt slightly to leaf texture without
   discarding what they already learned from ImageNet.

In [ ]:
base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
base_model.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
checkpoint = callbacks.ModelCheckpoint("best_head.h5", monitor="val_accuracy", save_best_only=True)

history_head = model.fit(
    train_ds, validation_data=val_ds, epochs=15,
    callbacks=[early_stop, reduce_lr, checkpoint],
)

In [ ]:
base_model.trainable = True
FINE_TUNE_FROM = len(base_model.layers) - 50
for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

checkpoint_ft = callbacks.ModelCheckpoint("best_finetuned.h5", monitor="val_accuracy", save_best_only=True)

history_finetune = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[early_stop, reduce_lr, checkpoint_ft],
)

## 6. Training curves

In [ ]:
def plot_history(histories, labels):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for h, label in zip(histories, labels):
        axes[0].plot(h.history["accuracy"], label=f"{label} train")
        axes[0].plot(h.history["val_accuracy"], label=f"{label} val", linestyle="--")
        axes[1].plot(h.history["loss"], label=f"{label} train")
        axes[1].plot(h.history["val_loss"], label=f"{label} val", linestyle="--")
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()


plot_history([history_head, history_finetune], ["head", "fine-tune"])

## 7. Evaluation — confusion matrix and classification report

In [ ]:
y_true, y_pred, y_conf = [], [], []
for images, labels_batch in test_ds:
    probs = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(probs, axis=1))
    y_conf.extend(np.max(probs, axis=1))

y_true, y_pred, y_conf = np.array(y_true), np.array(y_pred), np.array(y_conf)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — disease classifier")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

test_accuracy = (y_true == y_pred).mean()
print(f"Test accuracy: {test_accuracy:.4f}")

## 8. Confidence threshold for the disease classifier

Out-of-distribution and genuinely ambiguous inputs tend to produce a flatter, less peaked softmax
distribution than a confident, in-distribution prediction. Candidate thresholds are swept over the
**validation** set, not the test set, to avoid leaking the threshold decision into the reported
test metrics. The chosen value keeps most correct predictions while rejecting a meaningful share
of the classifier's mistakes.

In [ ]:
val_true, val_pred, val_conf = [], [], []
for images, labels_batch in val_ds:
    probs = model.predict(images, verbose=0)
    val_true.extend(np.argmax(labels_batch.numpy(), axis=1))
    val_pred.extend(np.argmax(probs, axis=1))
    val_conf.extend(np.max(probs, axis=1))

val_true, val_pred, val_conf = np.array(val_true), np.array(val_pred), np.array(val_conf)
val_correct = (val_true == val_pred)

candidate_thresholds = np.arange(0.50, 0.96, 0.05)
print(f"{'threshold':>10} {'kept_frac':>10} {'accuracy_on_kept':>18}")
for t in candidate_thresholds:
    kept = val_conf >= t
    kept_frac = kept.mean()
    acc_on_kept = val_correct[kept].mean() if kept.any() else float("nan")
    print(f"{t:>10.2f} {kept_frac:>10.2%} {acc_on_kept:>18.2%}")

CONFIDENCE_THRESHOLD = 0.65
print(f"\nSelected CONFIDENCE_THRESHOLD = {CONFIDENCE_THRESHOLD}")

## 9. Export labels and disease classifier

Files are written to a local `mango_model_output/` folder rather than a path relative to this
repository, since Colab and Kaggle Notebooks do not have this repository checked out — the
notebook's working directory there is unrelated to `backend/`. After this section runs, download
the folder's contents and copy `model.tflite`, `leaf_gate.tflite`, and `labels.json` into
`backend/app/ml/` in your local clone.

`labels.json` maps the softmax output index to a human-readable class name, so the backend never
needs the class list hardcoded in Python.

In [ ]:
ML_DIR = pathlib.Path("mango_model_output")
ML_DIR.mkdir(parents=True, exist_ok=True)
print("Writing outputs to:", ML_DIR.resolve())

labels = {str(i): name for i, name in enumerate(CLASS_NAMES)}
with open(ML_DIR / "labels.json", "w") as f:
    json.dump({
        "labels": labels,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
    }, f, indent=2)

print(json.dumps(labels, indent=2))

In [ ]:
model.save("mango_inception_v3.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(ML_DIR / "model.tflite", "wb") as f:
    f.write(tflite_model)

print("model.tflite size (MB):", len(tflite_model) / 1e6)

### Alternative: ONNX export

Not used for the deployed backend (see the compatibility and memory rationale above), but included
for completeness in case the serving target later moves to an ONNX Runtime-based host.

```python
import tf2onnx

spec = (tf.TensorSpec((None, *IMG_SIZE, 3), tf.float32, name="input"),)
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=13,
                                             output_path=str(ML_DIR / "model.onnx"))
```

## 10. Leaf gate — binary "is this a mango leaf?" classifier

**Positive class:** every image in the mango leaf disease dataset, disease label discarded — only
"is a mango leaf" matters here.

**Negative class:** a mix of (a) a different plant species' leaves, so the gate learns "mango
leaf" specifically rather than "leaf-shaped green object", and (b) generic non-leaf photos, so it
also rejects unrelated uploads outright. Both come from `tensorflow_datasets`, so this section
downloads its own data with no manual steps:

- Other-plant leaves: `tf_flowers` — a different plant, still organic and textured, a genuinely
  hard negative.
- Generic objects: `caltech101` — everyday object photos across 101 categories, standing in for
  "not a plant at all".

Both sources, and the mango leaf images, are loaded through `tf.data` pipelines rather than into a
single in-memory array, since materializing the full image set at once risks exhausting a Colab
free-tier instance's RAM once the dataset reaches a few thousand images.

In [ ]:
import tensorflow_datasets as tfds

GATE_IMG_SIZE = (224, 224)
GATE_BATCH_SIZE = 32

positive_paths = [str(p) for d in class_dirs for p in d.glob("*")]
random.shuffle(positive_paths)
n_positive = len(positive_paths)
n_negative_each = n_positive // 2


def load_positive(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, GATE_IMG_SIZE)
    return image, label


positive_ds = tf.data.Dataset.from_tensor_slices(
    (positive_paths, [1.0] * n_positive)
).map(load_positive, num_parallel_calls=tf.data.AUTOTUNE)


def to_negative_example(image, _original_label):
    return tf.image.resize(image, GATE_IMG_SIZE), tf.constant(0.0, dtype=tf.float32)


flowers_ds = (
    tfds.load("tf_flowers", split="train", as_supervised=True)
    .take(n_negative_each)
    .map(to_negative_example, num_parallel_calls=tf.data.AUTOTUNE)
)
objects_ds = (
    tfds.load("caltech101", split="train", as_supervised=True)
    .take(n_negative_each)
    .map(to_negative_example, num_parallel_calls=tf.data.AUTOTUNE)
)

print(f"Positives: {n_positive} | Negatives: {2 * n_negative_each} (tf_flowers + caltech101)")

Each source is split into train/val/test proportionally before combining, so the final splits stay
balanced across positive and negative sources without depending on `tf.data`'s buffered shuffle
covering the entire dataset.

In [ ]:
def split_dataset(dataset, n_total, val_frac=0.15, test_frac=0.15):
    n_val = int(n_total * val_frac)
    n_test = int(n_total * test_frac)
    test_split = dataset.take(n_test)
    val_split = dataset.skip(n_test).take(n_val)
    train_split = dataset.skip(n_test + n_val)
    return train_split, val_split, test_split


pos_train, pos_val, pos_test = split_dataset(positive_ds, n_positive)
flowers_train, flowers_val, flowers_test = split_dataset(flowers_ds, n_negative_each)
objects_train, objects_val, objects_test = split_dataset(objects_ds, n_negative_each)


def combine(*datasets):
    combined = datasets[0]
    for extra in datasets[1:]:
        combined = combined.concatenate(extra)
    return combined


gate_train_raw = combine(pos_train, flowers_train, objects_train)
gate_val_raw = combine(pos_val, flowers_val, objects_val)
gate_test_raw = combine(pos_test, flowers_test, objects_test)


def prep_gate(x, y):
    return mobilenet_preprocess(x), y


# A buffer sized to the full training set (~5,600 decoded 224x224 images) risks exhausting
# Colab's RAM on top of the disease classifier already loaded; a fixed buffer still shuffles
# well without holding the whole dataset in memory at once.
AUTOTUNE = tf.data.AUTOTUNE
GATE_SHUFFLE_BUFFER = 1000
gate_train_ds = (
    gate_train_raw.shuffle(GATE_SHUFFLE_BUFFER, seed=SEED)
    .map(prep_gate, num_parallel_calls=AUTOTUNE)
    .batch(GATE_BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
gate_val_ds = gate_val_raw.map(prep_gate, num_parallel_calls=AUTOTUNE).batch(GATE_BATCH_SIZE).prefetch(AUTOTUNE)
gate_test_ds = gate_test_raw.map(prep_gate, num_parallel_calls=AUTOTUNE).batch(GATE_BATCH_SIZE).prefetch(AUTOTUNE)

## 11. Leaf gate — model and training

`mobilenet_preprocess` is applied in the `tf.data` pipeline above, not as a layer inside
`gate_model` — the same pattern used for the disease classifier. `backend/app/services/inference.py`
applies the `/127.5 - 1` scaling itself before invoking either TFLite interpreter; baking the same
preprocessing into the model graph as well would scale each image twice and silently corrupt the
gate's predictions in production.

In [ ]:
import gc

# The disease classifier is already exported to disk (## 9) and isn't needed in memory anymore.
# Freeing it before training the gate model avoids GPU memory contention between the two models.
del model, base_model
gc.collect()
tf.keras.backend.clear_session()

In [ ]:
gate_base = MobileNetV2(weights="imagenet", include_top=False, input_shape=(*GATE_IMG_SIZE, 3))
gate_base.trainable = False

gate_inputs = layers.Input(shape=(*GATE_IMG_SIZE, 3))
x = gate_base(gate_inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
gate_outputs = layers.Dense(1, activation="sigmoid")(x)

gate_model = models.Model(gate_inputs, gate_outputs)
gate_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])

gate_early_stop = callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)

gate_history = gate_model.fit(
    gate_train_ds, validation_data=gate_val_ds, epochs=12,
    callbacks=[gate_early_stop],
)

## 12. Leaf gate evaluation

In [ ]:
gate_probs, gate_true = [], []
for images, labels_batch in gate_test_ds:
    probs = gate_model.predict(images, verbose=0).ravel()
    gate_probs.extend(probs)
    gate_true.extend(labels_batch.numpy())

gate_probs, gate_true = np.array(gate_probs), np.array(gate_true)
gate_preds = (gate_probs >= 0.5).astype(int)

gate_test_accuracy = (gate_preds == gate_true).mean()
print(classification_report(gate_true, gate_preds, target_names=["not_mango_leaf", "mango_leaf"]))
print(f"Leaf gate test accuracy: {gate_test_accuracy:.4f}")

gate_cm = confusion_matrix(gate_true, gate_preds)
plt.figure(figsize=(4, 4))
sns.heatmap(gate_cm, annot=True, fmt="d",
            xticklabels=["not_mango_leaf", "mango_leaf"], yticklabels=["not_mango_leaf", "mango_leaf"],
            cmap="Greens")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — leaf gate")
plt.tight_layout()
plt.show()

GATE_THRESHOLD = 0.5

In [ ]:
converter_gate = tf.lite.TFLiteConverter.from_keras_model(gate_model)
converter_gate.optimizations = [tf.lite.Optimize.DEFAULT]
gate_tflite_model = converter_gate.convert()

with open(ML_DIR / "leaf_gate.tflite", "wb") as f:
    f.write(gate_tflite_model)

print("leaf_gate.tflite size (MB):", len(gate_tflite_model) / 1e6)

with open(ML_DIR / "labels.json", "r") as f:
    labels_payload = json.load(f)

labels_payload["gate_threshold"] = GATE_THRESHOLD
labels_payload["gate_test_accuracy"] = float(gate_test_accuracy)
labels_payload["disease_test_accuracy"] = float(test_accuracy)

with open(ML_DIR / "labels.json", "w") as f:
    json.dump(labels_payload, f, indent=2)

print(json.dumps(labels_payload, indent=2))

## 13. Summary

| Model | File | Purpose | Threshold | Test accuracy |
|---|---|---|---|---|
| Leaf gate (MobileNetV2) | `leaf_gate.tflite` | Reject non-mango-leaf images before disease inference | sigmoid ≥ `GATE_THRESHOLD` (0.5) | see `gate_test_accuracy` in `labels.json` |
| Disease classifier (Inception V3) | `model.tflite` | Predict disease class among mango leaf classes | softmax top-1 ≥ `CONFIDENCE_THRESHOLD` (0.65) | see `disease_test_accuracy` in `labels.json` |

**Inference order enforced by the backend (`backend/app/services/inference.py`):**
1. Run the leaf gate. If its output is below `GATE_THRESHOLD`, reject immediately without calling
   the disease classifier.
2. Otherwise run the Inception V3 disease classifier. If its top-1 softmax probability is below
   `CONFIDENCE_THRESHOLD`, reject as not confident rather than returning a low-confidence label.
3. Otherwise return the predicted class name (from `labels.json`) and confidence score.

Both thresholds and both models' measured test accuracy are written into `labels.json`, so the
backend and this notebook remain in sync. After retraining, download `mango_model_output/` and
replace `model.tflite`, `leaf_gate.tflite`, and `labels.json` in `backend/app/ml/`.